In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin
import time

# Configuration
base_url = "https://junoschool.org"
visited_urls = set()
course_links = []
max_depth = 3
request_delay = 1
timeout = 15

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
}

def clean_text(text):
    if not text:
        return ""
    return re.sub(r'\s+', ' ', text).strip()

def is_valid_course_link(url, text):
    url_patterns = [r'course', r'program', r'learn', r'training']
    text_patterns = [r'course', r'program', r'certificate', r'learning']
    return any(re.search(p, url, re.I) for p in url_patterns) or any(re.search(p, text, re.I) for p in text_patterns)

def extract_courses_from_url(url, depth=0):
    try:
        if depth > max_depth:
            return

        print(f"🔍 Scanning ({depth}): {url}")
        time.sleep(request_delay)

        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()

        if 'text/html' not in response.headers.get('Content-Type', ''):
            return

        soup = BeautifulSoup(response.text, "html.parser")
        course_elements = []

        course_elements.extend(soup.find_all(class_=re.compile(r'course|program|card|item|listing|product', re.I)))
        course_elements.extend(soup.find_all('a', string=re.compile(r'course|program|learn|training|certificate', re.I)))
        course_elements.extend(soup.find_all(attrs={"itemtype": "http://schema.org/Course"}))

        for element in course_elements:
            link = element.find('a') if element.name != 'a' else element
            if not link or not link.has_attr('href'):
                continue

            href = link["href"]
            text = clean_text(link.get_text()) or clean_text(element.get_text())
            if not text or len(text) < 3:
                continue

            full_url = urljoin(base_url, href)
            if is_valid_course_link(full_url, text) and full_url.startswith(base_url):
                course_data = {
                    "Course Name": text[:200],
                    "Course Link": full_url,
                    "Source Page": url
                }

                try:
                    price_element = element.find(class_=re.compile(r'price|cost|fee', re.I))
                    if price_element:
                        course_data["Price"] = clean_text(price_element.get_text())

                    duration_element = element.find(class_=re.compile(r'duration|length|time', re.I))
                    if duration_element:
                        course_data["Duration"] = clean_text(duration_element.get_text())

                    desc_element = element.find(class_=re.compile(r'description|summary', re.I))
                    if desc_element:
                        course_data["Description"] = clean_text(desc_element.get_text())[:300]
                except:
                    pass

                if full_url not in [c["Course Link"] for c in course_links]:
                    course_links.append(course_data)

        if depth < max_depth:
            for link in soup.find_all("a", href=True):
                href = link["href"]
                if href.startswith('javascript:') or href.startswith('mailto:'):
                    continue

                full_url = urljoin(base_url, href)
                if full_url.startswith(base_url) and full_url not in visited_urls:
                    if re.search(r'\.(pdf|jpg|png|gif|zip|mp4)$', full_url, re.I):
                        continue

                    visited_urls.add(full_url)
                    if re.search(r'course|program|learn|training', full_url, re.I):
                        extract_courses_from_url(full_url, depth + 1)

    except requests.exceptions.RequestException as e:
        print(f"⚠️ Network error at {url}: {str(e)}")
    except Exception as e:
        print(f"⚠️ Error at {url}: {str(e)}")

# Start pages
start_urls = [
    base_url,
    urljoin(base_url, "/free-certificate-courses/"),
    urljoin(base_url, "/courses/"),
    urljoin(base_url, "/programs/"),
    urljoin(base_url, "/online-courses/"),
    urljoin(base_url, "/all-courses/")
]

for url in start_urls:
    if url not in visited_urls:
        visited_urls.add(url)
        extract_courses_from_url(url)

# Save to Excel at fixed path
if course_links:
    df = pd.DataFrame(course_links)
    df = df.sort_values("Course Name").drop_duplicates(subset=["Course Link"]).reset_index(drop=True)

    output_file = r"C:\Users\taslim.siddiqui\Downloads\Juno_School_All_Courses.xlsx"
    df.to_excel(output_file, index=False)

    print(f"\n✅ Extracted {len(df)} courses!")
    print(f"📁 Saved to: {output_file}")
    print(df.head(10).to_string(index=False))
else:
    print("❌ No courses found.")

    


🔍 Scanning (0): https://junoschool.org
🔍 Scanning (1): https://junoschool.org/tag/free-marketing-courses


KeyboardInterrupt: 

# JUNO CPP LINK EXTRACTION

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin

# Base configuration
base_url = "https://junoschool.org"
visited_urls = set()
course_links = []
headers = {'User-Agent': 'Mozilla/5.0'}

def is_course_url(url):
    """Check if URL is a course page and not a pagination or listing page"""
    url = url.lower()

    # Must contain these patterns
    course_patterns = [
        '/free-certificate-course/',
        '/course/',
        '/certificate/',
        '/training/',
        '/learn/'
    ]

    # Must NOT contain these patterns
    exclude_patterns = [
        '/tag/',
        '/category/',
        '/page/',
        '/author/',
        '/archive/',
        '/search/'
    ]

    # Check for at least one course pattern and no exclude patterns
    return (any(pattern in url for pattern in course_patterns) and
            not any(pattern in url for pattern in exclude_patterns) and
            not url.endswith(('/courses/', '/all-courses/')))

def extract_courses_from_url(url):
    try:
        print(f"🔍 Scanning: {url}")
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"⚠️ Failed to access {url} - Status code: {response.status_code}")
            return

        soup = BeautifulSoup(response.text, "html.parser")

        # Find all links that might be courses
        potential_links = soup.find_all("a", href=True)

        for a in potential_links:
            href = a["href"]
            text = a.get_text(strip=True)

            # Skip if no text or obviously not a course
            if not text or len(text) < 3 or text.lower() in ["home", "about", "contact", "login", "sign up"]:
                continue

            # Construct full URL
            full_url = urljoin(base_url, href)

            # Check if this is a course URL and not already collected
            if (full_url.startswith(base_url) and is_course_url(full_url) and 
                not any(x['Course Link'] == full_url for x in course_links)):

                # Clean the course name
                clean_name = re.sub(r'\s+', ' ', text).strip()

                # If the name is too short, try getting a better title from the page
                if len(clean_name) < 5:
                    title_tag = soup.find('h1') or soup.find('h2') or soup.find('title')
                    if title_tag:
                        clean_name = title_tag.get_text(strip=True)

                if clean_name:
                    course_links.append({"Course Name": clean_name, "Course Link": full_url})
                    print(f"✅ Found course: {clean_name} - {full_url}")

        # Queue more internal links for scanning
        internal_links = soup.find_all("a", href=re.compile(r"^/|^" + re.escape(base_url)))
        for link in internal_links:
            href = link["href"]
            full_internal_url = urljoin(base_url, href)

            # Skip already visited or irrelevant URLs
            if (full_internal_url.startswith(base_url) and
                full_internal_url not in visited_urls and
                not any(x in full_internal_url.lower() for x in ["wp-admin", "wp-login", "feed", ".jpg", ".png", ".pdf"]) and
                not re.search(r'/page/\d+', full_internal_url.lower())):

                visited_urls.add(full_internal_url)
                extract_courses_from_url(full_internal_url)

    except Exception as e:
        print(f"⚠️ Error processing {url}: {str(e)}")

# Start crawling from known pages
start_urls = [
    base_url,
    base_url + "/free-certificate-courses/",
    base_url + "/courses/",
    base_url + "/all-courses/",
    base_url + "/free-certificate-course/"
]

for url in start_urls:
    if url not in visited_urls:
        visited_urls.add(url)
        extract_courses_from_url(url)

# Save results to Excel
if course_links:
    df = pd.DataFrame(course_links)
    df = df.drop_duplicates(subset=["Course Link"]).reset_index(drop=True)

    output_file = "C:\\Users\\taslim.siddiqui\\Downloads\\Juno_School_All_Courses.xlsx"
    df.to_excel(output_file, index=False)

    print(f"\n✅ Successfully extracted {len(df)} unique courses!")
    print(f"📁 Saved to: {output_file}")
else:
    print("❌ No courses found. Please check the website structure.")


🔍 Scanning: https://junoschool.org
🔍 Scanning: https://junoschool.org/tag/free-marketing-courses
✅ Found course: Hyperpersonalized Marketing Strategies with AI - https://junoschool.org/free-certificate-course/hyperpersonalized-marketing-strategies-with-ai/
✅ Found course: Using Gamification in Marketing - https://junoschool.org/free-certificate-course/using-gamification-in-marketing/
✅ Found course: Personalization in Marketing - https://junoschool.org/free-certificate-course/personalization-in-marketing/
✅ Found course: Crafting Compelling Brand Stories - https://junoschool.org/free-certificate-course/crafting-compelling-brand-stories/
✅ Found course: The Power of Interactive Content - https://junoschool.org/free-certificate-course/the-power-of-interactive-content/
✅ Found course: Building a Brand Identity - https://junoschool.org/free-certificate-course/building-a-brand-identity/
✅ Found course: Master Writing Ad Copies - https://junoschool.org/free-certificate-course/master-writing-

# gitinfo.com CPPP LINK EXTRACTION


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin

# Base configuration
base_url = "https://gitinfo.com"
visited_urls = set()
course_links = []
headers = {'User-Agent': 'Mozilla/5.0'}

def is_course_url(url):
    """Check if URL is a course page based on GIT Academy structure"""
    url = url.lower()
    
    # Must contain these patterns
    course_patterns = [
        '/courses/',
        '/course/',
        '/training/',
        '/learn/',
        '/comptia-',
        '/ccna/',
        '/ccnp-',
        '/ceh-',
        '/chfi-',
        '/certified-',
        '/exam-'
    ]
    
    # Must NOT contain these patterns
    exclude_patterns = [
        '/tag/',
        '/category/',
        '/page/',
        '/author/',
        '/search/',
        '/wp-',
        '/feed/'
    ]
    
    return any(pattern in url for pattern in course_patterns) and \
           not any(pattern in url for pattern in exclude_patterns)

def extract_courses_from_page(url):
    try:
        print(f"🔍 Scanning: {url}")
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"⚠️ Failed to access {url} - Status code: {response.status_code}")
            return

        soup = BeautifulSoup(response.text, "html.parser")
        
        # Extract course title from the page
        title = soup.find('h1')
        if title:
            course_name = title.get_text(strip=True)
            
            # Only add if we don't already have this course
            if not any(x['Course Link'] == url for x in course_links):
                course_links.append({
                    "Course Name": course_name,
                    "Course Link": url
                })
                print(f"✅ Found course: {course_name} - {url}")

    except Exception as e:
        print(f"⚠️ Error processing {url}: {str(e)}")

def extract_courses_from_menu():
    try:
        print("🍔 Extracting courses from main menu...")
        response = requests.get(base_url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Find the main menu items
        menu_items = soup.select('#menu-main-menu > li.menu-item')
        
        for item in menu_items:
            # Check if this is a training courses menu item
            if 'Training Courses' in item.get_text():
                # Find all submenu items
                sub_menus = item.select('ul.gm-dropdown-menu--lvl-1 li.gm-menu-item a')
                
                for menu in sub_menus:
                    href = menu['href']
                    text = menu.get_text(strip=True)
                    
                    # Skip empty or non-course links
                    if not text or not href:
                        continue
                        
                    full_url = urljoin(base_url, href)
                    
                    # If this is a category page, we'll need to scan it
                    if '/courses/' in full_url.lower():
                        extract_courses_from_url(full_url)
                    else:
                        # Direct course link
                        if not any(x['Course Link'] == full_url for x in course_links):
                            course_links.append({
                                "Course Name": text,
                                "Course Link": full_url
                            })
                            print(f"✅ Found course: {text} - {full_url}")

    except Exception as e:
        print(f"⚠️ Error extracting menu: {str(e)}")

def extract_courses_from_url(url):
    try:
        if url in visited_urls:
            return
            
        visited_urls.add(url)
        print(f"🌐 Scanning category: {url}")
        
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Find course links on category pages
        course_anchors = soup.select('a[href*="/courses/"], a[href*="/course/"]')
        
        for a in course_anchors:
            href = a['href']
            text = a.get_text(strip=True)
            
            if not text or not href:
                continue
                
            full_url = urljoin(base_url, href)
            
            if is_course_url(full_url) and not any(x['Course Link'] == full_url for x in course_links):
                course_links.append({
                    "Course Name": text,
                    "Course Link": full_url
                })
                print(f"✅ Found course: {text} - {full_url}")

    except Exception as e:
        print(f"⚠️ Error processing category {url}: {str(e)}")

# Start scraping
extract_courses_from_menu()

# Also scan known important pages
important_pages = [
    base_url + "/courses/",
    base_url + "/courses/cisco/",
    base_url + "/courses/comptia/",
    base_url + "/courses/ec-council/",
    base_url + "/courses/certnexus/",
    base_url + "/courses/microsoft/",
    base_url + "/courses/cyber-security-master-program/"
]

for page in important_pages:
    extract_courses_from_url(page)

# Save results to Excel
if course_links:
    df = pd.DataFrame(course_links)
    df = df.drop_duplicates(subset=["Course Link"]).reset_index(drop=True)

    output_file = "C:\\Users\\taslim.siddiqui\\Downloads\\GIT_Academy_Courses.xlsx"
    df.to_excel(output_file, index=False)

    print(f"\n✅ Successfully extracted {len(df)} unique courses!")
    print(f"📁 Saved to: {output_file}")
else:
    print("❌ No courses found. Please check the website structure.")

🍔 Extracting courses from main menu...
🌐 Scanning category: https://gitinfo.com/courses/cisco/
✅ Found course: Cisco - https://gitinfo.com/courses/cisco/
✅ Found course: CCNA - https://gitinfo.com/courses/cisco/ccna/
✅ Found course: CCNP Collaboration - https://gitinfo.com/courses/cisco/ccnp-collaboration/
✅ Found course: CCNP Data Center - https://gitinfo.com/courses/cisco/ccnp-data-center/
✅ Found course: CCNP Enterprise - https://gitinfo.com/courses/cisco/ccnp-enterprise/
✅ Found course: CCNP Security - https://gitinfo.com/courses/cisco/ccnp-security/
✅ Found course: CCNP Service Provider - https://gitinfo.com/courses/cisco/ccnp-service-provider/
✅ Found course: Cisco Meraki Solution - https://gitinfo.com/courses/cisco/cisco-meraki-solution/
✅ Found course: CyberOps Associate - https://gitinfo.com/courses/cisco/cyberops-associate/
✅ Found course: CompTIA - https://gitinfo.com/courses/comptia/
✅ Found course: EC-Council - https://gitinfo.com/courses/ec-council/
✅ Found course: CertNe

# CPP 3ritechnologies

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

# Configure base URL and headers
base_url = "https://www.3ritechnologies.com"
headers = {'User-Agent': 'Mozilla/5.0'}
course_data = []

def scrape_courses():
    try:
        print(f"🌐 Connecting to {base_url}...")
        response = requests.get(base_url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # First try: Check if courses are listed in navigation menu
        print("🔍 Checking main navigation for courses...")
        nav_menu = soup.select('.menu-item a')
        for item in nav_menu:
            href = item.get('href', '')
            if '/course/' in href or '/training/' in href:
                course_name = item.get_text(strip=True)
                course_url = urljoin(base_url, href)
                if not any(c['url'] == course_url for c in course_data):
                    course_data.append({
                        'name': course_name,
                        'url': course_url
                    })
                    print(f"✅ Found in menu: {course_name} - {course_url}")
        
        # Second try: Check for course cards/grids
        print("🔍 Scanning for course cards...")
        course_cards = soup.select('.elementor-widget-wrap a, .course-card a, .eael-course a')
        for card in course_cards:
            href = card.get('href', '')
            if ('/course/' in href or '/training/' in href) and not href.endswith('/course/'):
                course_name = card.get_text(strip=True)
                if len(course_name) > 3:  # Filter out very short names
                    course_url = urljoin(base_url, href)
                    if not any(c['url'] == course_url for c in course_data):
                        course_data.append({
                            'name': course_name,
                            'url': course_url
                        })
                        print(f"✅ Found in cards: {course_name} - {course_url}")
        
        # Third try: Check dedicated course pages
        course_pages = [
            '/courses/',
            '/training/',
            '/programs/',
            '/it-courses/'
        ]
        
        for page in course_pages:
            print(f"🔍 Scanning {page} page...")
            page_url = urljoin(base_url, page)
            try:
                page_response = requests.get(page_url, headers=headers, timeout=10)
                if page_response.status_code == 200:
                    page_soup = BeautifulSoup(page_response.text, 'html.parser')
                    courses = page_soup.select('a[href*="/course/"], a[href*="/training/"]')
                    for course in courses:
                        href = course.get('href', '')
                        if not href.endswith(('/course/', '/courses/', '/training/')):
                            course_name = course.get_text(strip=True)
                            if len(course_name) > 3:
                                course_url = urljoin(base_url, href)
                                if not any(c['url'] == course_url for c in course_data):
                                    course_data.append({
                                        'name': course_name,
                                        'url': course_url
                                    })
                                    print(f"✅ Found on {page}: {course_name} - {course_url}")
            except Exception as e:
                print(f"⚠️ Error scanning {page_url}: {str(e)}")
        
        # Save results
        if course_data:
            df = pd.DataFrame(course_data)
            df = df.drop_duplicates(subset=['url'])
            output_path = "C:\\Users\\taslim.siddiqui\\Downloads\\3RI_Courses_List.xlsx"
            df.to_excel(output_path, index=False, columns=['name', 'url'])
            
            print("\n📋 Final Course List:")
            for course in course_data:
                print(f"{course['name']}\t{course['url']}")
                
            print(f"\n✅ Success! Found {len(df)} courses. Saved to {output_path}")
        else:
            print("❌ No courses found. The website structure may have changed.")
            
    except Exception as e:
        print(f"⚠️ Error: {str(e)}")

# Run the scraper
scrape_courses()

🌐 Connecting to https://www.3ritechnologies.com...
🔍 Checking main navigation for courses...
✅ Found in menu: Cyber Security - https://www.3ritechnologies.com/course/cyber-security-course-in-pune/
✅ Found in menu: Mastering in Data Science - https://3ritechnologies.com/course/data-science-online-training/
✅ Found in menu: Data Science with Python & Stats - https://3ritechnologies.com/course/data-science-with-python-and-machine-learning-certification-training/
✅ Found in menu: Mastering in Data Analytics - https://3ritechnologies.com/course/data-analytics-course-in-pune/
✅ Found in menu: STEP in Cloud Computing - https://3ritechnologies.com/course/amazon-web-services-course/
✅ Found in menu: AWS – Associate Solution Architect - https://3ritechnologies.com/course/aws-amazon-web-services/
✅ Found in menu: Azure - https://3ritechnologies.com/course/microsoft-azure-training-in-pune/
✅ Found in menu: STEP in DevOps Engineering - https://3ritechnologies.com/course/devops-foundation-certificat

# Eduonix LINK Extraction

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re
from collections import deque

base_url = "https://www.eduonix.com"
visited = set()
queue = deque([base_url])
headers = {'User-Agent': 'Mozilla/5.0'}
courses = []

# Regex for course-like page ending
course_url_regex = re.compile(r"https://www\.eduonix\.com/[a-z0-9\-]+$", re.I)

# Exclusion list for non-course pages
excluded = [
    "/login", "/signup", "/cart", "/checkout", "/dashboard", "/profile",
    "/terms", "/privacy", "/contact", "/about", "/faq", "/category",
    ".jpg", ".png", ".svg", ".pdf", "tel:", "mailto:", "javascript:", "#",
    "lifetime", "edegree", "infiniti", "deals", "freebies", "upcoming"
]

def is_course_page(url, text):
    return (
        text and len(text.strip()) > 5
        and url.startswith(base_url)
        and not any(x in url.lower() for x in excluded)
        and re.match(course_url_regex, url)
        and not url.endswith(('courses', 'course'))
    )

def should_crawl(url):
    return (
        url.startswith(base_url)
        and not any(x in url.lower() for x in excluded)
        and url not in visited
        and not url.endswith(('courses', 'course'))
    )

while queue and len(visited) < 100:  # Limit to 100 pages to prevent excessive crawling
    current_url = queue.popleft()
    visited.add(current_url)

    try:
        print(f"🔍 Scanning: {current_url}")
        r = requests.get(current_url, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        # Find all course links - looking for <a> tags with href and text
        for a in soup.find_all("a", href=True):
            href = urljoin(base_url, a["href"].strip())
            text = a.get_text(strip=True)
            
            if should_crawl(href):
                queue.append(href)
                
            if is_course_page(href, text):
                courses.append({
                    "Course Name": text,
                    "Course Link": href
                })
                
        # Special handling for course listing pages
        if "courses" in current_url:
            for course_card in soup.select('a[href*="/courses/"]'):
                href = urljoin(base_url, course_card["href"].strip())
                text = course_card.get_text(strip=True)
                
                if is_course_page(href, text):
                    courses.append({
                        "Course Name": text,
                        "Course Link": href
                    })

    except Exception as e:
        print(f"⚠️ Skipping {current_url}: {e}")

# Deduplicate
df = pd.DataFrame(courses).drop_duplicates(subset="Course Link").reset_index(drop=True)

# Save to Excel
output_path = "C:\\Users\\taslim.siddiqui\\Downloads\\Eduonix_Course_Links.xlsx"
df.to_excel(output_path, index=False)

print(f"\n✅ Extracted {len(df)} real course links.")
print(f"📁 Saved to: {output_path}")
print("\nSample of extracted courses:")
print(df.head(10).to_string(index=False))

🔍 Scanning: https://www.eduonix.com
🔍 Scanning: https://www.eduonix.com/logout
🔍 Scanning: https://www.eduonix.com/
🔍 Scanning: https://www.eduonix.com/
🔍 Scanning: https://www.eduonix.com/courses?track=5245940&utm_source=website&utm_medium=scroll_banner_1&utm_campaign=scroll_banner_1_28_05_2024
🔍 Scanning: https://www.eduonix.com/mighty-machine-learning-bundle
🔍 Scanning: https://www.eduonix.com/mighty-data-science-bundle
🔍 Scanning: https://www.eduonix.com/mighty-web-development-bundle
🔍 Scanning: https://www.eduonix.com/mighty-digital-marketing-bundle
🔍 Scanning: https://www.eduonix.com/courses?track=5245940&utm_source=eduonix&utm_medium=category_banner_ind&utm_campaign=category_banner_04_01_2024
🔍 Scanning: https://www.eduonix.com/business-data-analytics-bootcamp-master-ai-powered-insights
🔍 Scanning: https://www.eduonix.com/business-data-analytics-bootcamp-master-ai-powered-insights
🔍 Scanning: https://www.eduonix.com/business-data-analytics-bootcamp-master-ai-powered-insights
🔍 S

CPP CERTYBOX

In [ ]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin, urlparse
from concurrent.futures import ThreadPoolExecutor

# Configuration
BASE_URL = "https://www.certybox.com"
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
MAX_THREADS = 8

# Storage
course_data = []
visited_urls = set()

def is_course_page(url):
    """Strict check for course pages only (no categories)"""
    patterns = [
        r'^/courses/[^/]+/?$',
        r'^/course/[^/]+/?$',
        r'^/training/[^/]+/?$',
        r'^/certification/[^/]+/?$',
        r'^/program/[^/]+/?$'
    ]
    path = urlparse(url.lower()).path
    return any(re.match(pattern, path) for pattern in patterns)

def is_discovery_page(url):
    """Identify pages that may contain course links"""
    path = urlparse(url.lower()).path
    return not bool(re.search(r'course-category|category|topic|subject', path))

def extract_course_info(url):
    """Extract course name from its page"""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Try multiple ways to find course name
        name_selectors = [
            'h1.product_title', 'h1.course-title', 'h1.entry-title',
            'h1.title', 'h1.page-title', 'title'
        ]
        
        for selector in name_selectors:
            name = soup.select_one(selector)
            if name:
                course_name = re.sub(r'\s+', ' ', name.get_text(strip=True)).strip()
                return course_name
        
        return "Unknown Course Name"
    except:
        return "Unknown Course Name"

def process_page(url):
    """Process a page to find courses and their names"""
    if url in visited_urls:
        return
    visited_urls.add(url)
    
    print(f"Processing: {url}")
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Method 1: Course cards
        for card in soup.find_all(class_=re.compile(r'course|product|training|certification', re.I)):
            link = card.find('a', href=True)
            if link:
                full_url = urljoin(BASE_URL, link['href'])
                if is_course_page(full_url):
                    name = extract_course_name_from_card(card) or extract_course_info(full_url)
                    course_data.append({
                        "Course Name": name,
                        "Course Link": full_url
                    })
        
        # Method 2: Direct links
        for link in soup.find_all('a', href=True):
            full_url = urljoin(BASE_URL, link['href'])
            if is_course_page(full_url):
                name = link.get_text(strip=True)
                if len(name.split()) > 2:  # Filter out short/non-descriptive links
                    course_data.append({
                        "Course Name": re.sub(r'\s+', ' ', name).strip(),
                        "Course Link": full_url
                    })
        
        # Handle pagination
        pagination = soup.find(class_=re.compile(r'pagination|page-numbers', re.I))
        if pagination:
            for page in pagination.find_all('a', href=True):
                page_url = urljoin(BASE_URL, page['href'])
                if is_discovery_page(page_url) and page_url not in visited_urls:
                    process_page(page_url)
                    
    except Exception as e:
        print(f"Error processing {url}: {str(e)}")

def extract_course_name_from_card(card):
    """Extract name from course card element"""
    name = card.find(class_=re.compile(r'title|name|heading', re.I)) or card.find(['h2', 'h3', 'h4'])
    return name.get_text(strip=True) if name else None

def main():
    print("🚀 Starting CertyBox course extraction...")
    
    # Entry points that contain course listings
    entry_points = [
        BASE_URL + "/courses/",
    ]
    
    # Multi-threaded processing
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        executor.map(process_page, entry_points)
    
    # Process results
    if course_data:
        df = pd.DataFrame(course_data)
        df = df.drop_duplicates(subset=["Course Link"]).sort_values(by="Course Name")
        
        output_file = "C:\\Users\\taslim.siddiqui\\Downloads\\CertyBox_Courses_With_Links.xlsx"
        df.to_excel(output_file, index=False)
        
        print(f"\n✅ Success! Extracted {len(df)} courses")
        print(f"📁 Saved to: {output_file}")
        
        print("\n=== Sample Courses ===")
        print(df.head(10).to_string(index=False))
    else:
        print("❌ No courses found")

if __name__ == "__main__":
    main()

🚀 Starting CertyBox course extraction...
Processing: https://www.certybox.com/courses/
Processing: https://www.certybox.com/courses/page/2/
Processing: https://www.certybox.com/courses/page/3/
Processing: https://www.certybox.com/courses/page/4/
Processing: https://www.certybox.com/courses/page/5/
Processing: https://www.certybox.com/courses/page/6/
Processing: https://www.certybox.com/courses/page/7/
Processing: https://www.certybox.com/courses/page/8/
Processing: https://www.certybox.com/courses/page/9/
Processing: https://www.certybox.com/courses/page/10/
Processing: https://www.certybox.com/courses/page/11/
Processing: https://www.certybox.com/courses/page/12/
Processing: https://www.certybox.com/courses/page/13/
Processing: https://www.certybox.com/courses/page/14/
Processing: https://www.certybox.com/courses/page/15/

✅ Success! Extracted 118 courses
📁 Saved to: C:\Users\taslim.siddiqui\Downloads\CertyBox_Courses_With_Links.xlsx

=== Sample Courses ===
                           

# Henry harvin

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin

def extract_courses(html_content, base_url):
    """Extract ALL course names and links from HTML"""
    soup = BeautifulSoup(html_content, 'html.parser')
    courses = []
    
    # Find ALL <a> tags with href (hyperlinks)
    for link in soup.find_all('a', href=True):
        href = link['href'].strip()
        text = link.get_text(strip=True)
        
        # Skip if: Not a course link OR text too short
        if (not re.search(r'course|training|certification|program', href, re.I) 
            or len(text) < 3 
            or re.search(r'blog|category|testimonial|about|contact', href, re.I)):
            continue
        
        courses.append({
            "Course Name": re.sub(r'\s+', ' ', text),  # Clean whitespace
            "Course Link": urljoin(base_url, href)     # Make full URL
        })
    
    # Remove duplicates (same link or same name)
    df = pd.DataFrame(courses).drop_duplicates(subset=["Course Link", "Course Name"])
    return df.sort_values("Course Name").reset_index(drop=True)

def main():
    # Load HTML file
    with open("C:\\Users\\taslim.siddiqui\\Downloads\\1200+ Certification Courses with Gold Membership-Henry Harvin.html", "r", encoding="utf-8") as f:
        html = f.read()
    
    # Extract coursesa
    df = extract_courses(html, base_url="https://www.henryharvin.com")
    
    # Save to Excel
    output_path = r"C:\Users\taslim.siddiqui\Downloads\HenryHarvin_Courses_Complete.xlsx"
    df.to_excel(output_path, index=False)
    
    print(f"✅ Extracted {len(df)} courses!")
    print(f"📂 Saved to: {output_path}\n")
    print("Sample Courses:")
    print(df.head(10).to_string(index=False))

if __name__ == "__main__":
    main()

✅ Extracted 2178 courses!
📂 Saved to: C:\Users\taslim.siddiqui\Downloads\HenryHarvin_Courses_Complete.xlsx

Sample Courses:
                        Course Name                                                 Course Link
                    1-on-1 Training                 https://www.henryharvin.com/1-on-1-training
     10 Days Korean Language Course  https://www.henryharvin.com/10-days-korean-language-course
                3D VFX (C3V) Course                   https://www.henryharvin.com/3d-vfx-course
                                4.6 https://www.coursereport.com/schools/henry-harvin-education
                          5S Course                https://www.henryharvin.com/5s-certification
                  7 QC Tools Course             https://www.henryharvin.com/7-qc-tools-training
                  8051 & AVR Course                 https://www.henryharvin.com/8051-avr-course
              8051 & Arduino Course             https://www.henryharvin.com/8051-arduino-course
            

# Global Institute of Regulatory

In [11]:
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from collections import defaultdict

def extract_courses_from_boxes(soup, base_url):
    link_to_name = {}
    for box in soup.select('.single-causes-box'):
        link = box.select_one('h4 a')
        if link:
            course_name = link.get_text(strip=True)
            course_link = urljoin(base_url, link['href']).strip()
            if course_link not in link_to_name:
                link_to_name[course_link] = course_name
    return link_to_name

def extract_courses_from_footer(soup, base_url):
    link_to_name = {}
    for footer_section in soup.select('.f-widget'):
        if "Our Courses" in footer_section.get_text():
            for link in footer_section.select('a[href]'):
                course_name = link.get_text(strip=True)
                course_link = urljoin(base_url, link['href']).strip()
                if course_link not in link_to_name:
                    link_to_name[course_link] = course_name
    return link_to_name

def extract_courses_from_courses_page(soup, base_url):
    link_to_name = {}
    for link in soup.select('a[href]'):
        course_name = link.get_text(strip=True)
        href = link.get('href')
        if href and course_name and (
            '/course' in href or '/diploma' in href or '/training' in href or '/validation' in href
        ):
            course_link = urljoin(base_url, href).strip()
            if course_link not in link_to_name:
                link_to_name[course_link] = course_name
    return link_to_name

def collect_all_links(soup, base_url):
    all_links = set()
    for a in soup.select('a[href]'):
        href = urljoin(base_url, a['href']).strip()
        all_links.add(href)
    return all_links

def find_links_with_multiple_names(soup, base_url):
    link_names = defaultdict(list)
    for a in soup.select('a[href]'):
        href = urljoin(base_url, a['href']).strip()
        name = a.get_text(strip=True)
        if href and name:
            link_names[href].append(name)
    
    for link, names in link_names.items():
        if len(set(names)) > 1:
            print(f"⚠️ Link with multiple names: {link} -> {set(names)}")

def extract_all_courses(html_content, base_url):
    soup = BeautifulSoup(html_content, 'html.parser')
    combined = {}

    # Combine from all sources
    sources = [
        extract_courses_from_boxes(soup, base_url),
        extract_courses_from_footer(soup, base_url),
        extract_courses_from_courses_page(soup, base_url),
    ]

    for source in sources:
        for link, name in source.items():
            if link not in combined:
                combined[link] = name

    # Log multiple names for same link
    find_links_with_multiple_names(soup, base_url)

    # Log all found links for audit
    all_links = collect_all_links(soup, base_url)
    print(f"🔍 Total unique hrefs found on page (for audit): {len(all_links)}")

    # Convert to DataFrame
    df = pd.DataFrame([(name, link) for link, name in combined.items()],
                      columns=["Course Name", "Course Link"])
    df = df.sort_values("Course Name").reset_index(drop=True)
    return df

def main():
    file_path = "C:\\Users\\taslim.siddiqui\\Downloads\\Global Institute of Regulatory Affairs-Pharmaceutical Training Courses.html"
    base_url = "https://www.regulatoryinstitute.com/"
    
    with open(file_path, "r", encoding="utf-8") as f:
        html = f.read()
    
    df = extract_all_courses(html, base_url)

    print(f"\n✅ Found {len(df)} unique course links:")
    print(df.to_string(index=False))

    output_path = "C:\\Users\\taslim.siddiqui\\Downloads\\GIRA_Courses_UniqueByLink.xlsx"
    df.to_excel(output_path, index=False)
    print(f"\n📂 Saved to: {output_path}")

if __name__ == "__main__":
    main()


⚠️ Link with multiple names: https://www.regulatoryinstitute.com/ -> {'About Us', 'Regulatory Affairs', 'Apply Online', 'Registration', 'Courses'}
⚠️ Link with multiple names: https://www.regulatoryinstitute.com/diploma-reg-affairs.html -> {'Part Time Course', 'Part time course', 'Read More'}
⚠️ Link with multiple names: https://www.regulatoryinstitute.com/ra-distance-learning.html -> {'DRA-Distance Learning', 'RA- Distance Learning', 'Read More'}
⚠️ Link with multiple names: https://www.regulatoryinstitute.com/eCTD-training.html -> {'ECTD-Training', 'Read More', 'eCTD Training'}
⚠️ Link with multiple names: https://www.regulatoryinstitute.com/pharmacovigilance-course.html -> {'PHARMACOVIGILANCE Course', 'Read More', 'Pharmacovigilance Course', 'PHARMACOVIGILANCE SCIENTIST TRAINING COURSE'}
⚠️ Link with multiple names: https://www.regulatoryinstitute.com/qaqc-diploma.html -> {'QA/QC Program', 'Read More', 'QA - QC Program'}
⚠️ Link with multiple names: https://www.regulatoryinstitute.c

In [40]:
import pandas as pd
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin

def extract_courses(html_content, base_url):
    """Extract ALL course names and links from HTML with improved parsing"""
    soup = BeautifulSoup(html_content, 'html.parser')
    courses = []
    
    # Find all course sections - they're in elementor-column elements
    course_sections = soup.find_all(class_='elementor-column')
    
    for section in course_sections:
        # Get course name from icon-box description
        description = section.find(class_='elementor-icon-box-description')
        if not description:
            continue
            
        course_name = description.get_text(strip=True)
        if not course_name or len(course_name) < 5:
            continue
            
        # Find the associated "Know More" link
        know_more = None
        buttons = section.find_all('a', href=True)
        
        for button in buttons:
            if 'know more' in button.get_text().lower():
                know_more = button
                break
        
        # If no "Know More" found, take first valid link
        if not know_more and buttons:
            know_more = buttons[0]
        
        if know_more:
            href = know_more['href'].strip()
            if href and not href.startswith('#') and not href.startswith('javascript'):
                # Clean course name
                course_name = re.sub(r'\s+', ' ', course_name).strip()
                
                courses.append({
                    "Course Name": course_name,
                    "Course Link": urljoin(base_url, href)
                })
    
    # Create DataFrame
    df = pd.DataFrame(courses) if courses else pd.DataFrame(columns=["Course Name", "Course Link"])
    
    # Remove duplicates and sort
    df = df.drop_duplicates(subset=["Course Name", "Course Link"])
    df = df.sort_values("Course Name").reset_index(drop=True)
    
    return df

def main():
    try:
        # Load HTML file
        with open("C:\\Users\\taslim.siddiqui\\Downloads\\Courses List – iPEC Solutions Private Limited.html", "r", encoding="utf-8") as f:
            html = f.read()
        
        # Extract courses
        df = extract_courses(html, base_url="https://ipecsolutions.com")
        
        # Save to Excel
        output_path = "C:\\Users\\taslim.siddiqui\\Downloads\\iPEC_Solutions_Courses.xlsx"
        df.to_excel(output_path, index=False)
        
        print(f"✅ Extracted {len(df)} courses!")
        print(f"📂 Saved to: {output_path}\n")
        print("All Courses:")
        print(df.to_string(index=False))
        
    except Exception as e:
        print(f"❌ Error occurred: {str(e)}")

if __name__ == "__main__":
    main()

✅ Extracted 12 courses!
📂 Saved to: C:\Users\taslim.siddiqui\Downloads\iPEC_Solutions_Courses.xlsx

All Courses:
                                       Course Name                                                                      Course Link
                                     AI in Medical                                    https://ipecsolutions.com/ai-in-medical-field
Advanced Certified Professional inAI -Data Science                                           https://ipecsolutions.com/data-science
                           Business Data Analytics                                https://ipecsolutions.com/business-data-analytics
                       Data Analysis for Educators                            https://ipecsolutions.com/data-analysis-for-educators
                                    Data Analytics                                https://ipecsolutions.com/business-data-analytics
            DevOps, AWS, Linux, Networking(4 in 1) https://ipecsolutions.com/devops-aws-network

In [5]:
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def extract_all_courses(html_content, base_url):
    soup = BeautifulSoup(html_content, "html.parser")
    courses = []

    # Extract from icon-box-description (original method)
    for desc in soup.select(".elementor-icon-box-description"):
        course_name = desc.get_text(separator=" ", strip=True)
        if not course_name or course_name.lower() in ['top courses', 'fast track learning programmes']:
            continue

        parent_section = desc.find_parent(class_="elementor-widget")
        if not parent_section:
            continue

        link_tag = parent_section.find_next("a", class_="elementor-button", href=True)
        if link_tag:
            course_link = urljoin(base_url, link_tag["href"])
            
            # Try to find rating if available
            rating_tag = parent_section.find_next(class_=lambda x: x and "/5" in x)
            rating = rating_tag.get_text(strip=True) if rating_tag else ""
            
            courses.append({
                "Course Name": course_name,
                "Course Link": course_link,
                "Rating": rating
            })

    # Additional extraction for heading titles (new method)
    for heading in soup.select(".elementor-heading-title.elementor-size-default"):
        course_name = heading.get_text(separator=" ", strip=True)
        if not course_name or course_name.lower() in ['top courses', 'fast track learning programmes']:
            continue

        parent_section = heading.find_parent(class_="elementor-widget")
        if not parent_section:
            continue

        link_tag = parent_section.find_next("a", class_="elementor-button", href=True)
        course_link = urljoin(base_url, link_tag["href"]) if link_tag else ""
        
        # Try to find rating if available
        rating_tag = parent_section.find_next(class_=lambda x: x and "/5" in x)
        rating = rating_tag.get_text(strip=True) if rating_tag else ""
        
        courses.append({
            "Course Name": course_name,
            "Course Link": course_link,
            "Rating": rating
        })

    # Remove duplicates based on Course Name and Link
    df = pd.DataFrame(courses)
    df = df.drop_duplicates(subset=["Course Name", "Course Link"])
    
    # Clean rating column - keep only valid ratings (like "4.9/5")
    df['Rating'] = df['Rating'].apply(lambda x: x if isinstance(x, str) and "/5" in x else "")
    
    # Sort by Course Name
    df = df.sort_values("Course Name").reset_index(drop=True)
    return df

def main():
    html_file = "C:\\Users\\taslim.siddiqui\\Downloads\\Courses List – iPEC Solutions Private Limited.html"
    output_file = "C:\\Users\\taslim.siddiqui\\Downloads\\iPEC_Solutions_Courses_Clean.xlsx"

    with open(html_file, "r", encoding="utf-8") as f:
        html = f.read()

    base_url = "https://ipecsolutions.com"
    df = extract_all_courses(html, base_url)

    if df.empty:
        print("⚠️ No course data found.")
    else:
        df.to_excel(output_file, index=False)
        print(f"✅ Extracted {len(df)} courses.")
        print(f"📂 Saved to: {output_file}")
        print("📊 Sample:")
        print(df.head(10).to_string(index=False))

if __name__ == "__main__":
    main()

✅ Extracted 66 courses.
📂 Saved to: C:\Users\taslim.siddiqui\Downloads\iPEC_Solutions_Courses_Clean.xlsx
📊 Sample:
                                        Course Name                                                                                                                  Course Link Rating
                                              4.9/5 https://ipecsolutions.com#elementor-action%3Aaction%3Dpopup%3Aopen%26settings%3DeyJpZCI6IjE3MzgiLCJ0b2dnbGUiOmZhbHNlfQ%3D%3D       
                                            AI & ML https://ipecsolutions.com#elementor-action%3Aaction%3Dpopup%3Aopen%26settings%3DeyJpZCI6IjE3MzgiLCJ0b2dnbGUiOmZhbHNlfQ%3D%3D       
                                      AI in Medical https://ipecsolutions.com#elementor-action%3Aaction%3Dpopup%3Aopen%26settings%3DeyJpZCI6IjE3MzgiLCJ0b2dnbGUiOmZhbHNlfQ%3D%3D       
                                      AI in Medical                                                                                https://ipecsoluti

In [74]:
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def extract_all_courses(html_content, base_url):
    soup = BeautifulSoup(html_content, "html.parser")
    courses = []

    # Loop through every icon-box-description (course name)
    for desc in soup.select(".elementor-icon-box-description"):
        course_name = desc.get_text(separator=" ", strip=True)
        if not course_name or course_name.lower() in ['top courses', 'fast track learning programmes']:
            continue

        # Find the nearest "Know More" button within the same grandparent block
        parent_section = desc.find_parent(class_="elementor-widget")
        if not parent_section:
            continue

        link_tag = parent_section.find_next("a", class_="elementor-button", href=True)
        if link_tag:
            course_link = urljoin(base_url, link_tag["href"])
            courses.append({
                "Course Name": course_name,
                "Course Link": course_link
            })

    # Remove duplicates
    df = pd.DataFrame(courses)
    df = df.drop_duplicates(subset="Course Link").sort_values("Course Name").reset_index(drop=True)
    return df

def main():
    html_file = "C:\\Users\\taslim.siddiqui\\Downloads\\Courses List – iPEC Solutions Private Limited.html"
    output_file = "C:\\Users\\taslim.siddiqui\\Downloads\\iPEC_Solutions_Courses_Clean.xlsx"

    with open(html_file, "r", encoding="utf-8") as f:
        html = f.read()

    base_url = "https://ipecsolutions.com"
    df = extract_all_courses(html, base_url)

    if df.empty:
        print("⚠️ No course data found.")
    else:
        df.to_excel(output_file, index=False)
        print(f"✅ Extracted {len(df)} courses.")
        print(f"📂 Saved to: {output_file}")
        print("📊 Sample:")
        print(df.head(10).to_string(index=False))

if __name__ == "__main__":
    main()


✅ Extracted 11 courses.
📂 Saved to: C:\Users\taslim.siddiqui\Downloads\iPEC_Solutions_Courses_Clean.xlsx
📊 Sample:
                                        Course Name                                                                                                                  Course Link
                                      AI in Medical                                                                                https://ipecsolutions.com/ai-in-medical-field
Advanced Certified Professional in AI -Data Science https://ipecsolutions.com#elementor-action%3Aaction%3Dpopup%3Aopen%26settings%3DeyJpZCI6IjE3MzgiLCJ0b2dnbGUiOmZhbHNlfQ%3D%3D
                            Business Data Analytics                                                                            https://ipecsolutions.com/business-data-analytics
                        Data Analysis for Educators                                                                        https://ipecsolutions.com/data-analysis-for-educators


Nextford CPP price extract

In [9]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Load your Excel file
input_file = "C:\\Users\\taslim.siddiqui\\Downloads\\111111111.xlsx"  # ✅ Change path as needed
df = pd.read_excel(input_file)

# Make sure 'Course_link' exists
if 'Course Link' not in df.columns:
    raise ValueError("Column 'Course_link' not found in Excel.")

# List to store extracted tuition prices
tuition_prices = []

# Headers for requests
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Loop through each row and extract price
for index, row in df.iterrows():
    url = row['Course Link']
    print(f"🔍 Processing: {url}")
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")

        tuition_section = soup.find("h3", string="Monthly Tuition")
        if tuition_section:
            parent_div = tuition_section.find_parent("div", class_="num")
            box_div = parent_div.find_parent("div", class_="box")
            price_div = box_div.find("div", class_="text")
            price = price_div.text.strip() if price_div else "❌ Price not found"
        else:
            price = "❌ 'Monthly Tuition' not found"
    except Exception as e:
        price = f"❌ Error: {str(e)}"

    tuition_prices.append(price)

# Add new column to original DataFrame
df['Monthly_Tuition'] = tuition_prices

# Save result to Excel
output_file = "C:\\Users\\taslim.siddiqui\\Downloads\\nexford_courses_with_prices.xlsx"
df.to_excel(output_file, index=False)
print(f"\n✅ All data (including 'cpp' column) saved to: {output_file}")


🔍 Processing: https://www.nexford.edu/courses/blockchain-for-finance
🔍 Processing: https://www.nexford.edu/courses/auditing-and-assurance-services
🔍 Processing: https://www.nexford.edu/courses/founders-financing-and-legal
🔍 Processing: https://www.nexford.edu/courses/digital-marketing-fundamentals
🔍 Processing: https://www.nexford.edu/courses/roadmap-to-success
🔍 Processing: https://www.nexford.edu/courses/digitizing-supply-chain-management
🔍 Processing: https://www.nexford.edu/courses/business-communication-managers
🔍 Processing: https://www.nexford.edu/courses/business-environment
🔍 Processing: https://www.nexford.edu/courses/doing-business-in-china
🔍 Processing: https://www.nexford.edu/courses/international-business-and-culture
🔍 Processing: https://www.nexford.edu/courses/organizational-strategy
🔍 Processing: https://www.nexford.edu/courses/marketing-strategy-and-planning
🔍 Processing: https://www.nexford.edu/courses/doing-business-in-india
🔍 Processing: https://www.nexford.edu/cou

In [17]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Input file path
input_file = "C:\\Users\\taslim.siddiqui\\Downloads\\upgrad.xlsx"
df = pd.read_excel(input_file)

# Make sure column exists
if 'Course link' not in df.columns:
    raise ValueError("Excel must contain a 'Course link' column.")

# Results list
total_prices = []

# Headers for the request
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Process each URL
for index, row in df.iterrows():
    url = row['Course link']
    print(f"🔍 Scraping: {url}")
    
    total_price = "❌ Total not found"
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.content, "html.parser")

        # First attempt: <h3 class="ProgramPricingWithSpecialization_price__MSWen">
        h3_tag = soup.find("h3", class_="ProgramPricingWithSpecialization_price__MSWen")
        if h3_tag:
            total_price = h3_tag.get_text(strip=True).split(" ")[0] + " " + h3_tag.get_text(strip=True).split(" ")[1]
        else:
            # Second attempt: <span class="font-medium text-bodySmall text-greyscale-main ...">
            span_tag = soup.find("span", class_="font-medium text-bodySmall text-greyscale-main ml-spacing4 md:-tracking-0.16 xs:-tracking-0.14")
            if span_tag:
                total_price = span_tag.get_text(strip=True)
    
    except Exception as e:
        total_price = f"❌ Error: {str(e)}"
    
    total_prices.append(total_price)
    print(f"✅ Total Price: {total_price}")  # Print the output for each course

# Add results to DataFrame
df['Total_Price'] = total_prices

# Save to Excel
output_file = "C:\\Users\\taslim.siddiqui\\Downloads\\upgrad_courses_with_prices.xlsx"
df.to_excel(output_file, index=False)
print(f"\n✅ All done. Results saved to: {output_file}")


🔍 Scraping: https://cdn.upgrad.com/UpGrad/temp/96663e32-af77-453f-a920-47e57574fd9b/AC+in+Brand+Communication+Management.pdf


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


✅ Total Price: ❌ Total not found
🔍 Scraping: https://www.upgrad.com/advanced-general-management-programme-imt-ghaziabad/
✅ Total Price: INR 1,53,000*
🔍 Scraping: https://www.upgrad.com/bba-180-ects-iu-germany/
✅ Total Price: ❌ Total not found
🔍 Scraping: https://upgradcampus.com/courses/digital-marketing/?utm_source=uGwebsite and utm_medium=uG_CoursesMenu and utm_campaign=ugc_dm/
✅ Total Price: ❌ Total not found
🔍 Scraping: https://www.upgrad.com/doctor-of-business-administration-from-esgci/
✅ Total Price: INR 8,14,000*
🔍 Scraping: https://www.upgrad.com/human-resource-management-epgp-liba/
✅ Total Price: INR 1,55,000*
🔍 Scraping: https://www.upgrad.com/professional-certificate-effective-leadership-management-msu/
✅ Total Price: ❌ Total not found
🔍 Scraping: https://www.upgrad.com/management-academy-imt-joblink/
✅ Total Price: ❌ Total not found
🔍 Scraping: https://www.upgrad.com/law-llm-corporate-and-finance-jgu/
✅ Total Price: INR 4,00,000*
🔍 Scraping: https://www.upgrad.com/managemen

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


✅ Total Price: ❌ Total not found
🔍 Scraping: https://www.upgrad.com/certification-program-in-digital-marketing-upgradcampus/
✅ Total Price: ❌ Total not found
🔍 Scraping: https://www.upgrad.com/machine-learning-ai-pgd-iiitb-lp/
✅ Total Price: ❌ Total not found
🔍 Scraping: https://www.upgrad.com/digital-marketing-and-communication-pgc-mica/
✅ Total Price: INR 84,999*
🔍 Scraping: https://www.upgrad.com/brand-and-communication-management-certificate-program-mica/
✅ Total Price: INR 84,999*
🔍 Scraping: https://www.upgrad.com/digital-marketing-and-communication-pgc-mica/
✅ Total Price: INR 84,999*
🔍 Scraping: https://www.upgrad.com/digital-marketing-and-communication-pgc-mica/
✅ Total Price: INR 84,999*
🔍 Scraping: https://www.upgrad.com/digital-marketing-and-communication-pgc-mica/
✅ Total Price: INR 84,999*
🔍 Scraping: https://www.upgrad.com/brand-and-communication-management-certificate-program-mica/
✅ Total Price: INR 84,999*
🔍 Scraping: https://cdn.upgrad.com/UpGrad/temp/96663e32-af77-4

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


✅ Total Price: ❌ Total not found
🔍 Scraping: https://www.upgrad.com/digital-marketing-and-communication-pgc-mica/
✅ Total Price: INR 84,999*
🔍 Scraping: https://www.upgrad.com/digital-marketing-and-communication-pgc-mica/
✅ Total Price: INR 84,999*
🔍 Scraping: https://www.upgrad.com/brand-and-communication-management-certificate-program-mica/
✅ Total Price: INR 84,999*
🔍 Scraping: https://www.upgrad.com/digital-marketing-and-communication-pgc-mica/
✅ Total Price: INR 84,999*

✅ All done. Results saved to: C:\Users\taslim.siddiqui\Downloads\upgrad_courses_with_prices.xlsx


In [20]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

# Step 1: Load course links from Excel
input_file = "C:\\Users\\taslim.siddiqui\\Downloads\\Henry_Harvin _input file.xlsx"
df = pd.read_excel(input_file)

# Step 2: Prepare results list
results = []

# Step 3: Loop through links and scrape clean price
for link in df['Course_link']:
    try:
        response = requests.get(link, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Find the <h6> tag with data-group-value="course-fee"
        price_tag = soup.find("h6", {"data-group-value": "course-fee"})

        if price_tag:
            full_text = price_tag.get_text(strip=True)
            # Extract only numeric price part (e.g., ₹ 2,35,000 → 2,35,000)
            match = re.search(r"₹?\s?[\d,]+", full_text)
            price = match.group().replace("₹", "").strip() if match else "Price not found"
        else:
            price = "Price not found"

        results.append({'Course Link': link, 'Price': price})
        print(f"✅ {link} -> {price}")

    except Exception as e:
        print(f"❌ Error fetching {link}: {e}")
        results.append({'Course Link': link, 'Price': "Error"})

# Step 4: Save cleaned results to Excel
output_file = "C:\\Users\\taslim.siddiqui\\Downloads\\course_prices_TIMME PRO.xlsx"
pd.DataFrame(results).to_excel(output_file, index=False)
print(f"📁 Saved to '{output_file}'")


✅ https://timespro.com/executive-education/iim-kozhikode-professional-certificate-programme-in-the-art-of-leadership -> 86,500
✅ https://timespro.com/early-career/post-graduate-programme-in-logistics-supply-chain-management -> 85,000
✅ https://timespro.com/early-career/post-graduate-certificate-in-data-science-and-machine-learning-applications -> 1,19,000
✅ https://timespro.com/executive-education/mica-the-emerging-cmo-a-certificate-programme-for-senior-marketing-professionals -> 2,10,000
✅ https://timespro.com/executive-education/mica-executive-certificate-programme-in-strategic-communication-with-storytelling -> 1,10,000
✅ https://timespro.com/executive-education/k-j-somaiya-institute-of-management-mba-for-working-executives -> 3,45,000
✅ https://timespro.com/executive-education/imt-hyderabad-executive-certificate-programme-in-advanced-management -> 1,70,000
✅ https://timespro.com/executive-education/iit-delhi-executive-programme-for-advanced-product-management -> 1,69,000
✅ https://

In [91]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Step 1: Load course links from Excel
input_file ="C:\\Users\\taslim.siddiqui\\OneDrive - Eduvanz Financing Private Limited\\Documents\\LINK EXTRACTION\\LINK EXTRACTION TIMEPRO.xlsx"
df = pd.read_excel(input_file)

# Step 2: Prepare results list
results = []

# Step 3: Loop through links and scrape full price text
for link in df['Course_link']:
    try:
        response = requests.get(link, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Find the <h6> tag with data-group-value="course-fee"
        price_tag = soup.find("h6", {"data-group-value": "course-fee"})
        price = price_tag.get_text(strip=True) if price_tag else "Price not found"

        results.append({'Course Link': link, 'Price': price})
        print(f"✅ {link} -> {price}")

    except Exception as e:
        print(f"❌ Error fetching {link}: {e}")
        results.append({'Course Link': link, 'Price': "Error"})

# Step 4: Save cleaned results to Excel
output_file = "C:\\Users\\taslim.siddiqui\\Downloads\\course_prices_TIMME PRO.xlsx"
pd.DataFrame(results).to_excel(output_file, index=False)
print(f"📁 Saved to '{output_file}'")


✅ https://timespro.com/executive-education/iim-kozhikode-professional-certificate-programme-in-the-art-of-leadership -> ₹ 86,500
✅ https://timespro.com/early-career/post-graduate-programme-in-logistics-supply-chain-management -> ₹85,000
✅ https://timespro.com/early-career/post-graduate-certificate-in-data-science-and-machine-learning-applications -> ₹1,19,000+ Taxes
✅ https://timespro.com/executive-education/mica-the-emerging-cmo-a-certificate-programme-for-senior-marketing-professionals -> ₹ 2,10,000 + Taxes
✅ https://timespro.com/executive-education/mica-executive-certificate-programme-in-strategic-communication-with-storytelling -> ₹ 1,10,000 + Taxes
✅ https://timespro.com/executive-education/k-j-somaiya-institute-of-management-mba-for-working-executives -> ₹ 3,45,000
✅ https://timespro.com/executive-education/imt-hyderabad-executive-certificate-programme-in-advanced-management -> ₹ 1,70,000
✅ https://timespro.com/executive-education/iit-delhi-executive-programme-for-advanced-produc

In [41]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import urllib3

# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def extract_original_price(url):
    headers = {
        'User-Agent': 'Mozilla/5.0'
    }

    try:
        response = requests.get(url, headers=headers, verify=False, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')

        original_price_tag = soup.find('span', class_='disc-amt')
        if original_price_tag:
            return original_price_tag.get_text(strip=True)
        else:
            return "❌ Not Found"
    except Exception as e:
        return f"❌ Error: {e}"

# === Step 1: Load Excel with course URLs ===
input_path = "C:\\Users\\taslim.siddiqui\\Downloads\\Croma campus.xlsx"   # Change this
output_path = "C:\\Users\\taslim.siddiqui\\Downloads\\Croma campusprice.xlsx"  # Change this

df = pd.read_excel(input_path)

# === Step 2: Extract prices for each link ===
df['Original Price'] = df['Course link'].apply(extract_original_price)

# === Step 3: Save output ===
df.to_excel(output_path, index=False)

print("✅ Extraction completed and saved to:", output_path)


✅ Extraction completed and saved to: C:\Users\taslim.siddiqui\Downloads\Croma campusprice.xlsx


In [47]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

# Load course links from Excel
df = pd.read_excel("C:\\Users\\taslim.siddiqui\\Downloads\\indigo learn.xlsx")  # Ensure file is in the same directory or provide full path

# Setup Chrome driver (headless mode)
options = webdriver.ChromeOptions()
options.add_argument("--headless")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Function to extract original price
def extract_original_price(url):
    try:
        driver.get(url)
        time.sleep(5)  # wait for page to load

        # Find all strike prices
        prices = driver.find_elements(By.CLASS_NAME, "tss-1d4sff5-strikePrice")
        if prices:
            return prices[0].text.strip()
        else:
            return "❌ Price not found"
    except Exception as e:
        return f"❌ Error: {str(e)}"

# Loop through links and get prices
results = []
for link in df['Course link']:
    price = extract_original_price(link)
    print(f"{link} --> {price}")
    results.append({"Course_link": link, "Original_Price": price})

# Save results to Excel
output_df = pd.DataFrame(results)
output_df.to_excel("C:\\Users\\taslim.siddiqui\\Downloads\\indigolearn_prices_output.xlsx", index=False)

driver.quit()


Fundamentals of Financial Accounting & Cost Accounting (New): Prepare for Paper 2 (indigolearn.com) --> ❌ Error: Message: invalid argument
  (Session info: chrome=138.0.7204.159)
Stacktrace:
	GetHandleVerifier [0x0xad1af3+62339]
	GetHandleVerifier [0x0xad1b34+62404]
	(No symbol) [0x0x911f80]
	(No symbol) [0x0x9026f5]
	(No symbol) [0x0x900da9]
	(No symbol) [0x0x9014fb]
	(No symbol) [0x0x915b4e]
	(No symbol) [0x0x9a1367]
	(No symbol) [0x0x97f3bc]
	(No symbol) [0x0x9a07a3]
	(No symbol) [0x0x97f1b6]
	(No symbol) [0x0x94e7a2]
	(No symbol) [0x0x94f644]
	GetHandleVerifier [0x0xd46683+2637587]
	GetHandleVerifier [0x0xd41a8a+2618138]
	GetHandleVerifier [0x0xaf856a+220666]
	GetHandleVerifier [0x0xae8998+156200]
	GetHandleVerifier [0x0xaef12d+182717]
	GetHandleVerifier [0x0xad9a38+94920]
	GetHandleVerifier [0x0xad9bc2+95314]
	GetHandleVerifier [0x0xac4d0a+9626]
	BaseThreadInitThunk [0x0x76295d49+25]
	RtlInitializeExceptionChain [0x0x770bd1ab+107]
	RtlGetAppContainerNamedObjectPath [0x0x770bd131+5

In [56]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# 🔧 Function to extract course price
def get_expertrons_course_price(url):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")  # Run silently
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    try:
        driver.get(url)
        time.sleep(5)  # Allow JS to load

        # Locate the price span
        price_elem = driver.find_element(By.CLASS_NAME, "course-price")
        course_price = price_elem.text.strip()
        return course_price if course_price else "❌ Price not found"

    except Exception as e:
        return f"❌ Error"
    finally:
        driver.quit()

# 🔁 Load Excel with course links
input_file = "C:\\Users\\taslim.siddiqui\\Downloads\\Experton_input.xlsx"  # 🔁 Change path as needed
df = pd.read_excel(input_file)

# 📦 Store prices
results = []

for link in df['Course link']:
    print(f"Checking: {link}")
    price = get_expertrons_course_price(link)
    results.append({'Course_link': link, 'Course Price': price})

# 💾 Save to Excel
output_df = pd.DataFrame(results)
output_file = "C:\\Users\\taslim.siddiqui\\Downloads\\expertrons_prices_output.xlsx"  # 🔁 Change as needed
output_df.to_excel(output_file, index=False)

print("✅ All done! Saved to:", output_file)


Checking: https://programs.expertrons.com/pgcpbfsiwebinar/?SourceCampaign=webapp&SourceMedium=header
Checking: https://programs.expertrons.com/pg-diploma-bankingandfinance/?utm_source=organic and utm_medium=homepage and utm_campaign=website
Checking: https://programs.expertrons.com/pg-diploma-in-banking-and-finance/?SourceCampaign=webapp&SourceMedium=header
Checking: https://programs.expertrons.com/wp-content/uploads/2023/08/Expertrons_PLUS.pdf
Checking: https://programs.expertrons.com/wp-content/uploads/2023/08/Foxmula_by_Expertrons.pdf
Checking: https://programs.expertrons.com/wp-content/uploads/2024/05/Expertrons-Speak-Easy.pdf
Checking: https://www.expertrons.com/advanced-hr-analytics-certification-program/?SourceCampaign=webapp&SourceMedium=header
Checking: https://www.expertrons.com/advanced-human-resources-certification-program-iit/?utm_source=organic and utm_medium=homepage and utm_campaign=website
Checking: https://www.expertrons.com/app/courses/5fe0776d68d29542e8dfcdb9?Source

In [67]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

# === Step 1: Define the fee extractor function ===
def get_all_aima_fees(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    try:
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.content, 'html.parser')

        fee_results = []

        # 1. Fees from <td><p> with Rs.
        td_tags = soup.find_all('td', {'valign': 'top', 'width': '312'})
        for td in td_tags:
            p_tag = td.find('p')
            if p_tag and 'Rs.' in p_tag.text:
                fee_results.append(p_tag.text.strip())

        # 2. <span> like "Online- Live Classes-" followed by <p> Rs.
        spans = soup.find_all('span')
        for span in spans:
            if 'Live Classes' in span.text or 'Face to Face' in span.text:
                next_p = span.find_next('p')
                if next_p and 'Rs.' in next_p.text:
                    label = span.text.strip()
                    price = next_p.text.strip()
                    fee_results.append(f"{label} {price}")

        # 3. <strong> with 'Program fee'
        strong_tags = soup.find_all('strong')
        for tag in strong_tags:
            if re.search(r'Program\s+fee', tag.text, re.I):
                fee_results.append(tag.text.strip())

        return ' | '.join(fee_results) if fee_results else '❌ Fee Not Found'

    except Exception as e:
        return f"❌ Error: {str(e)}"

# === Step 2: Load input Excel and process links ===
input_path = r"C:\Users\taslim.siddiqui\Downloads\All India Management Association (AIMA)_input.xlsx"
output_path = r"C:\Users\taslim.siddiqui\Downloads\All India Management Association (AIMA)_price.xlsx"

# Load course links from Excel
df = pd.read_excel(input_path)

# Add output column
df['Extracted Fee(s)'] = df['Course link'].apply(get_all_aima_fees)

# Save to output Excel
df.to_excel(output_path, index=False)

print("✅ Price extraction completed and saved to output Excel.")


✅ Price extraction completed and saved to output Excel.


In [3]:
import undetected_chromedriver as uc
import time
import re

# Setup undetected Chrome
options = uc.ChromeOptions()
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36")

driver = uc.Chrome(options=options)

# 🔗 Replace this with your target course link
url = "https://www.henryharvin.com/schedule/korean-language-course"

driver.get(url)
time.sleep(8)  # Wait for JS to load fully

html = driver.page_source

# 🎯 Extract course name and batch price from the page source
price_match = re.search(r'"batch_price"\s*:\s*"(\d+)"', html)
course_match = re.search(r'"course_name"\s*:\s*"([^"]+)"', html)

if course_match and price_match:
    course_name = course_match.group(1)
    price_inr = price_match.group(1)
else:
    course_name = "Not Found"
    price_inr = "Not Found"

driver.quit()

# 📋 Print the result
print("✅ Course Name:", course_name)
print("💰 Batch Price (INR):", price_inr)
print("🔗 URL:", url)


✅ Course Name: Korean Language Course
💰 Batch Price (INR): 198000
🔗 URL: https://www.henryharvin.com/schedule/korean-language-course


In [8]:
import pandas as pd
import undetected_chromedriver as uc
import time
import re

# 📥 Read input Excel
input_file = "C:\\Users\\taslim.siddiqui\\Downloads\\Henry_Harvin _input file.xlsx"  # ⬅️ Update path
df = pd.read_excel(input_file)

# 🔄 Prepare result storage
results = []

# ⚙️ Setup undetected Chrome
options = uc.ChromeOptions()
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36")

driver = uc.Chrome(options=options)

# 🔁 Loop through course links
for idx, row in df.iterrows():
    url = row['Course_link']
    print(f"Processing: {url}")
    
    try:
        driver.get(url)
        time.sleep(8)  # Wait for JS to load

        html = driver.page_source

        # 🎯 Regex to extract course info
        price_match = re.search(r'"batch_price"\s*:\s*"(\d+)"', html)
        course_match = re.search(r'"course_name"\s*:\s*"([^"]+)"', html)

        course_name = course_match.group(1) if course_match else "Not Found"
        price_inr = price_match.group(1) if price_match else "Not Found"

    except Exception as e:
        course_name = "Error"
        price_inr = "Error"
        print(f"⚠️ Error processing {url}: {e}")

    # 📋 Append results
    results.append({
        "Course Name": course_name,
        "Batch Price (INR)": price_inr,
        "Course Link": url
    })

# ✅ Quit browser
driver.quit(200)

# 💾 Save to Excel
output_df = pd.DataFrame(results)
output_file = "C:\\Users\\taslim.siddiqui\\Downloads\\henryharvin_course_prices.xlsx"  # ⬅️ Update path
output_df.to_excel(output_file, index=False)

print("✅ Extraction complete! File saved at:", output_file)


Processing: https://animation.henryharvin.com/graphic-design
Processing: https://animation.henryharvin.com/social-media-visual-design
Processing: https://aviation.henryharvin.com/airport-ground-service
Processing: https://aviation.henryharvin.com/aviation-hospitality-travel-and-customer-service
Processing: https://aviation.henryharvin.com/hospitality-management
Processing: https://henryharvin.com/certificate-course-in-advertising-and-marketing
Processing: https://henryharvin.com/delta-course
Processing: https://henryharvin.com/email-marketing-course
Processing: https://henryharvin.com/english-speaking-course
Processing: https://henryharvin.com/first-step-for-kids-course-in-korean
Processing: https://henryharvin.com/french-language-course-for-kids
Processing: https://henryharvin.com/frm-course
Processing: https://henryharvin.com/german-language-course
Processing: https://henryharvin.com/journalism-and-mass-communication-content-writing-course
Processing: https://henryharvin.com/korean-l

KeyboardInterrupt: 

In [10]:
import undetected_chromedriver as uc
import time
from bs4 import BeautifulSoup
import re

# Setup undetected Chrome
options = uc.ChromeOptions()
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36")

driver = uc.Chrome(options=options)

# Course URL
url = "https://www.henryharvin.com/french-language-course"
driver.get(url)
time.sleep(8)  # Wait for page to fully load

# Get page source
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

# Extract course name
course_name_match = re.search(r'"course_name"\s*:\s*"([^"]+)"', html)
course_name = course_name_match.group(1) if course_name_match else "Not Found"

# Extract all syllabus modules
syllabus_data = []

modules = soup.find_all("p")
for p_tag in modules:
    if p_tag.find("strong") and "Module" in p_tag.text:
        module_title = p_tag.get_text(strip=True)
        next_sibling = p_tag.find_next_sibling()
        module_content = []

        # Add the <p> tag content after <strong>
        description = next_sibling.get_text(strip=True) if next_sibling and next_sibling.name == "p" else ""
        if description:
            module_content.append(description)

        # Find corresponding <ul> or <ol> list items
        ul = next_sibling.find_next_sibling("ul") if next_sibling else None
        if ul:
            for li in ul.find_all("li"):
                module_content.append("- " + li.get_text(strip=True))

        syllabus_data.append(f"{module_title}\n" + "\n".join(module_content))

driver.quit()

# Print result
print("✅ Course Name:", course_name)
print("🔗 Course Link:", url)
print("\n📘 Course Syllabus:")
print("\n\n".join(syllabus_data))


✅ Course Name: French Language Course
🔗 Course Link: https://www.henryharvin.com/french-language-course

📘 Course Syllabus:
Module 1: Explorer la langue!
In this module, participants will get to know about the language right from the basics.
- Getting to know about the language
- Alphabet / Pronounciation
- Numbers

Module 2: Les Verbes!
This module will help the candidate to conjugate the verbs and make simple sentences.
- Introduction to Verbs
- Verb Group Conjugations

Module 3: La Communication
This module will have the student to get through the basics of basic french communication skills.
- Greetings
- Days of the week
- Months of the Year
- Colours

Module 4: Construire des phrases
This module will help the candidate with solid sentence-building skills.
- Positive and Negative Phrases
- Adjectives
- Interrogatives + Possessive + Demonstrative

Module 5: Talents d'orateur
This module will help students to enhance their question-making skills.
- Question-Making Skills

Module 6: A